In [1]:
#!/usr/bin/env python

import os
# Silence icechunk rust warnings. Must be set before importing icechunk.
os.environ.setdefault("RUST_LOG", "error")

import icechunk
from virtualizarr.parsers.hdf.hdf import _construct_manifest_array

import obstore
from obstore.store import HTTPStore

import io
import logging
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import xarray as xr
import zarr
from zarr.codecs import Zlib
from tqdm import tqdm

logging.basicConfig(level=logging.INFO)
log = logging.getLogger(__name__)

# Constants
NLAT, NLON = 1800, 3600          # HG grid dimensions
CLAT = 72                        # latitude chunk size (matches source HDF5)

# Updated fill values
FILL_MISSING = np.int16(-9999)   # fill value for missing retrievals
FILL_NOOBS = np.int16(-9999) 
#FILL_MISSING  = np.int16(-32768)   # cloudy / no retrieval
#FILL_NOOBS    = np.int16(-32767)   # ocean / no observation

SCALE_FACTOR = 0.1               # raw × 0.1 → cm

GPORTAL_BASE = (
    "https://gportal.jaxa.jp/download/standard/GCOM-W/GCOM-W.AMSR2"
    "/L3.SND_10/2/{yyyy}/{mm}/"
)
FNAME_TMPL = "GW1AM2_{date}_01D_{orbit}_L3SGSNDHG2210210.h5"
ORBIT_CODES = ["EQMA", "EQMD"]
ORBIT_LABELS = ["Ascending", "Descending"]

# CF time reference
TIME_UNITS = "days since 1970-01-01"
TIME_CALENDAR = "proleptic_gregorian"
TIME_EPOCH = pd.Timestamp("1970-01-01")

GPORTAL_URL = "https://gportal.jaxa.jp/"


def gportal_url(date: pd.Timestamp, orbit: str) -> str:
    """Return the G-Portal HTTPS URL for a date and orbit code."""
    return GPORTAL_BASE.format(
        yyyy=f"{date.year:04d}", mm=f"{date.month:02d}"
    ) + FNAME_TMPL.format(
        date=date.strftime("%Y%m%d"), orbit=orbit
    )


def _stream_hdf5(path: str, access_options: dict | None = None):
    """Return a context manager containing an HDF5 HTTP object in memory."""
    from urllib.parse import urlsplit

    if access_options is None:
        access_options = {}

    parts = urlsplit(path)
    base_url = f"{parts.scheme}://{parts.netloc}"
    obj_path = parts.path.lstrip("/")
    store = HTTPStore.from_url(base_url, **access_options)
    data = obstore.get(store, obj_path).bytes()
    buf = io.BytesIO(bytes(data))

    class _CM:
        def __enter__(self):
            return buf

        def __exit__(self, *_):
            buf.close()

    return _CM()


def _av(value):
    """Convert an HDF5 attribute value to a JSON-serialisable value."""
    if isinstance(value, np.ndarray):
        flat = value.flatten()
        if flat.dtype.kind in ("S", "U", "O"):
            items = [
                item.decode() if isinstance(item, bytes) else str(item)
                for item in flat
            ]
            return items[0] if len(items) == 1 else items
        values = flat.tolist()
        return values[0] if len(values) == 1 else values
    return value.decode() if isinstance(value, bytes) else value


def _get_manifests(chunk_url: str) -> tuple:
    """Download an HDF5 file and extract its VirtualiZarr manifest."""
    from urllib.parse import urlsplit

    parts = urlsplit(chunk_url)
    base_url = f"{parts.scheme}://{parts.netloc}"
    obj_path = parts.path.lstrip("/")
    store = HTTPStore.from_url(base_url)
    buf = io.BytesIO(obstore.get(store, obj_path).bytes())

    try:
        with h5py.File(buf, "r") as hdf:
            geophysical_data = hdf["Geophysical Data"]
            manifest_array = _construct_manifest_array(
                chunk_url, geophysical_data, "/"
            )
            attrs = {
                key: _av(value)
                for key, value in geophysical_data.attrs.items()
            }
        return manifest_array.manifest, attrs
    finally:
        buf.close()


def initialize_store(
    session: icechunk.Session,
    commit_msg: str = "Initialize empty store",
) -> None:
    """Create the arrays in a new IceChunk repository."""
    lats = np.linspace(89.95, -89.95, NLAT).astype("float32")
    lons = np.linspace(0.05, 359.95, NLON).astype("float32")

    root = zarr.open_group(session.store, mode="w", zarr_format=3)

    root.require_array(
        "time",
        shape=(0,),
        chunks=(512,),
        dtype="int32",
        dimension_names=["time"],
        attributes={
            "units": TIME_UNITS,
            "calendar": TIME_CALENDAR,
            "long_name": "observation date",
        },
    )

    root.require_array(
        "orbit",
        shape=(2,),
        chunks=(2,),
        dtype=str,
        dimension_names=["orbit"],
        attributes={"long_name": "orbit direction"},
    )
    root["orbit"][:] = ORBIT_LABELS

    root.require_array(
        "band",
        shape=(2,),
        chunks=(2,),
        dtype=str,
        dimension_names=["band"],
        attributes={"long_name": "band name"},
    )
    root["band"][:] = ["snow_depth", "quality_flag"]

    root.require_array(
        "lat",
        shape=(NLAT,),
        chunks=(NLAT,),
        dtype="float32",
        dimension_names=["lat"],
        attributes={
            "units": "degrees_north",
            "long_name": "latitude",
        },
    )
    root["lat"][:] = lats

    root.require_array(
        "lon",
        shape=(NLON,),
        chunks=(NLON,),
        dtype="float32",
        dimension_names=["lon"],
        attributes={
            "units": "degrees_east",
            "long_name": "longitude",
        },
    )
    root["lon"][:] = lons

    root.require_array(
        "geophysical_data",
        shape=(0, 2, NLAT, NLON, 2),
        chunks=(1, 1, CLAT, NLON, 2),
        dtype="int16",
        fill_value=int(FILL_MISSING),
        compressors=[Zlib(level=9)],
        dimension_names=["time", "orbit", "lat", "lon", "band"],
        attributes={
            "long_name": (
                "geophysical data (band 0 = snow depth, "
                "band 1 = quality flag)"
            ),
            "units": "cm",
            "scale_factor": SCALE_FACTOR,
            "_FillValue": int(FILL_MISSING),
            "no_observation_value": int(FILL_NOOBS),
            "comment": (
                f"No observation value {int(FILL_NOOBS)} indicates ocean or "
                "areas without observation. This value is preserved in the "
                "raw data but is not treated as a fill value by CF conventions."
            ),
        },
    )

    session.commit(commit_msg)
    log.info("Initialized empty store")


def stored_days(repo: icechunk.Repository) -> list[int]:
    """Return stored day values in their current storage order."""
    session = repo.readonly_session("main")
    root = zarr.open_group(session.store, mode="r", zarr_format=3)
    return root["time"][:].tolist()


def insert_date(date: pd.Timestamp, repo: icechunk.Repository) -> None:
    """Insert a date using virtual references to the source HDF5 chunks."""
    day_val = int((date - TIME_EPOCH).days)
    date_iso = date.strftime("%Y-%m-%d")
    current = stored_days(repo)

    if day_val in current:
        slot = current.index(day_val)
        log.info("%s → skipped (already present, slot %d)", date_iso, slot)
        return

    orbit_manifests: dict[int, object] = {}
    for orbit_index, orbit in enumerate(ORBIT_CODES):
        chunk_url = gportal_url(date, orbit)
        try:
            manifest, _ = _get_manifests(chunk_url)
            orbit_manifests[orbit_index] = manifest
        except Exception as exc:
            log.warning(
                "  could not fetch %s for %s: %s",
                orbit,
                date_iso,
                exc,
            )

    if not orbit_manifests:
        log.info("%s → skipped (no orbits available on G-Portal)", date_iso)
        return

    new_slot = len(current)
    session = repo.writable_session("main")
    root = zarr.open_group(session.store, mode="r+", zarr_format=3)

    root["time"].resize((new_slot + 1,))
    root["time"][new_slot] = day_val
    root["geophysical_data"].resize((new_slot + 1, 2, NLAT, NLON, 2))

    # Each source chunk has shape (CLAT, NLON, 2), so both bands are
    # represented by the same virtual source chunk.
    for orbit_index, manifest in orbit_manifests.items():
        for (lat_chunk, lon_chunk, band_chunk), reference in manifest.iter_refs():
            key = (
                f"geophysical_data/c/{new_slot}/{orbit_index}/"
                f"{lat_chunk}/{lon_chunk}/{band_chunk}"
            )
            session.store.set_virtual_ref(
                key,
                reference["path"],
                offset=reference["offset"],
                length=reference["length"],
            )

    session.commit(f"Insert {date_iso} → slot {new_slot}")


def select_time_range(ds: xr.Dataset, start: str, end: str) -> xr.Dataset:
    """
    Select a time interval without requiring a sorted or complete time index.

    Dates are appended to the repository, so an existing repository may not
    have monotonically ordered time values. Boolean selection avoids pandas'
    restriction on slicing a non-monotonic DatetimeIndex.
    """
    times = pd.DatetimeIndex(pd.to_datetime(ds["time"].values))
    mask = (times >= pd.Timestamp(start)) & (times <= pd.Timestamp(end))
    return ds.isel(time=np.flatnonzero(mask))

# Open or create the IceChunk store.
try:
    repo = icechunk.Repository.open(
        icechunk.s3_storage(
            bucket="airborne-smce-prod-user-bucket",
            prefix="JOIN/icechunk-stores/GCOM-W1-AMSR2-L3-SND",
        ),
        authorize_virtual_chunk_access={GPORTAL_URL: icechunk.credentials.HttpAccess},
    )
    log.info("Found existing icechunk store")
except icechunk.IcechunkError as exc:
    log.warning(
        "Failed to open existing repo with error: %s. "
        "Trying to create a fresh repo.",
        exc,
    )
    config = icechunk.RepositoryConfig.default()
    config.set_virtual_chunk_container(
        icechunk.VirtualChunkContainer(GPORTAL_URL, icechunk.http_store())
    )
    repo = icechunk.Repository.create(
        icechunk.s3_storage(
            bucket="airborne-smce-prod-user-bucket",
            prefix="JOIN/icechunk-stores/GCOM-W1-AMSR2-L3-SND",
        ),
        config=config,
        authorize_virtual_chunk_access={GPORTAL_URL: icechunk.credentials.HttpAccess},
    )
    initialize_store(repo.writable_session("main"))


date_seq = pd.date_range(
    start="2019-01-01",
    end="2019-01-01",
    freq="D",
)

for date in tqdm(date_seq):
    insert_date(date, repo)


log.info("Testing reading a subset of data")
session = repo.readonly_session("main")
ds = xr.open_zarr(session.store)

#LIS requires the missing values to be -9999
ds['geophysical_data'] = ds['geophysical_data'].fillna(FILL_MISSING)

# Spatial, orbit, and band selection can use normal label selection. Time is
# selected with a boolean index because the repository may contain dates in
# insertion order rather than chronological order.
sub = ds.sel(lat=52.07, lon=-91.99, method="nearest").sel(orbit="Descending", band="snow_depth")
#sub = ds.sel(lat=0., lon=-91.99, method="nearest").sel(orbit="Descending", band="snow_depth")
sub = select_time_range(sub, "2019-01-01", "2019-01-01")

print(sub["geophysical_data"].values)

INFO:__main__:Found existing icechunk store
100%|██████████| 1/1 [00:00<00:00,  7.34it/s]
INFO:__main__:Testing reading a subset of data
/home/conda/dgiles/eda67c72-1791236921-417-data_access/lib/python3.14/site-packages/xarray/conventions.py:210: SerializationWarning: variable 'geophysical_data' has multiple fill values {np.int64(-32768), np.int64(-32767)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)


StorageError: error fetching virtual reference: Operation not supported: Range request not supported by download/standard/GCOM-W/GCOM-W.AMSR2/L3.SND_10/2/2019/01/GW1AM2_20190101_01D_EQMD_L3SGSNDHG2210210.h5

context:
   0: icechunk::store::get
           with key="geophysical_data/c/122/1/5/0/0" byte_range=From(0)
             at icechunk/src/store.rs:183

In [2]:
print(ds)
print(ds['geophysical_data'].attrs)

for key, value in ds['geophysical_data'].attrs.items():
    print(f"{key}: {value}")

<xarray.Dataset> Size: 644GB
Dimensions:           (time: 3104, orbit: 2, lat: 1800, lon: 3600, band: 2)
Coordinates:
  * time              (time) datetime64[ns] 25kB 2018-09-01 ... 2026-07-01
  * orbit             (orbit) StringDType() 32B 'Ascending' 'Descending'
  * lat               (lat) float32 7kB 89.95 89.85 89.75 ... -89.85 -89.95
  * lon               (lon) float32 14kB 0.05 0.15 0.25 ... 359.8 359.9 360.0
  * band              (band) StringDType() 32B 'snow_depth' 'quality_flag'
Data variables:
    geophysical_data  (time, orbit, lat, lon, band) float64 644GB dask.array<chunksize=(1, 1, 72, 3600, 2), meta=np.ndarray>
{'long_name': 'geophysical data (band 0 = snow depth, band 1 = quality flag)', 'units': 'cm'}
long_name: geophysical data (band 0 = snow depth, band 1 = quality flag)
units: cm


In [14]:
print(ds.info())

xarray.Dataset {
dimensions:
	time = 547 ;
	orbit = 2 ;
	lat = 1800 ;
	lon = 3600 ;
	band = 2 ;

variables:
	float64 geophysical_data(time, orbit, lat, lon, band) ;
		geophysical_data:long_name = geophysical data (band 0 = snow depth, band 1 = quality flag) ;
		geophysical_data:units = cm ;
	object band(band) ;
		band:long_name = band name ;
	float32 lat(lat) ;
		lat:units = degrees_north ;
		lat:long_name = latitude ;
	float32 lon(lon) ;
		lon:units = degrees_east ;
		lon:long_name = longitude ;
	object orbit(orbit) ;
		orbit:long_name = orbit direction ;
	datetime64[ns] time(time) ;
		time:long_name = observation date ;

// global attributes:
}None
